# Food Waste Prediction - Data Preprocessing

This notebook performs data cleaning, feature engineering, and preprocessing for the Food Waste Prediction System.

### Objectives
- Load the training dataset
- Clean and standardize the data
- Handle missing values
- Perform feature engineering
- Separate features and target
- Split the data into training and testing sets
- Build a preprocessing pipeline for numerical and categorical features

## 1. Import Libraries and Load Dataset

Import the required libraries and load the training dataset from the project dataset directory.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../dataset/train.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (911, 12)


,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg
0,0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465
1,1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,NaN,MeAt,51.549053
2,4,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,Beginner,MeAt,53.008323
3,5,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,Beginner,MeAt,48.621527
4,6,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,Intermediate,MEAT,44.156984


## 2. Initial Data Inspection

Inspect the dataset structure, data types, missing values, and basic information before applying preprocessing steps.

In [2]:
df.info()

print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                911 non-null    int64  
 1   date              911 non-null    object 
 2   meals_served      911 non-null    int64  
 3   kitchen_staff     911 non-null    int64  
 4   temperature_C     911 non-null    float64
 5   humidity_percent  911 non-null    float64
 6   day_of_week       911 non-null    int64  
 7   special_event     911 non-null    int64  
 8   past_waste_kg     911 non-null    float64
 9   staff_experience  747 non-null    object 
 10  waste_category    911 non-null    object 
 11  food_waste_kg     911 non-null    float64
dtypes: float64(4), int64(5), object(3)
memory usage: 85.5+ KB

Missing values:
ID                    0
date                  0
meals_served          0
kitchen_staff         0
temperature_C         0
humidity_percent      0
day_of_week  

## 3. Data Cleaning

The dataset is cleaned by removing the identifier column and standardizing categorical values. Missing values are retained at this stage and will be handled automatically within the preprocessing pipeline.

In [3]:
# Remove identifier
df = df.drop(columns=["ID"])

# Standardize categorical values
df["staff_experience"] = (
    df["staff_experience"]
    .astype("object")
    .str.strip()
    .str.lower()
)

df["waste_category"] = (
    df["waste_category"]
    .astype("object")
    .str.strip()
    .str.lower()
)

## 5. Separate Features and Target

The target variable is `food_waste_kg`, which represents the amount of food waste in kilograms. The remaining columns are used as input features.

In [4]:
X = df.drop(columns=["food_waste_kg"])
y = df["food_waste_kg"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (911, 10)
Target: (911,)


## 6. Train-Test Split

The dataset is divided into training and testing sets. 80% of the data is used for training and 20% is reserved for evaluating the model on unseen data.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (728, 10)
Testing set: (183, 10)


## 7. Outlier Treatment

`01_eda.ipynb` (Section 6, Outlier Detection) identified IQR-based outliers in
`meals_served` (24 records, 2.63%) and `temperature_C` (22 records, 2.41%),
with bounds of approximately **[-83.00, 701.00]** for `meals_served` and
**[-3.99, 48.48]** for `temperature_C` on the full dataset. The EDA
recommended these be "investigated and handled carefully during
preprocessing" rather than dropped, since some may reflect genuine
high-volume cafeteria days.

Outliers in the **target** (`food_waste_kg`, also 24 records / 2.63% per EDA)
are deliberately left untouched here - capping the target is a modeling
decision (e.g. a log-transform), not a feature-preprocessing one.

Bounds below are recalculated on `X_train` only (not the full dataset, unlike
the EDA notebook which used all 911 rows) to avoid any leakage from the test
split, then applied to cap both `X_train` and `X_test`.

In [6]:
def get_iqr_bounds(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - factor * iqr, q3 + factor * iqr

outlier_columns = ["meals_served", "temperature_C"]
outlier_bounds = {}

for col in outlier_columns:
    lower, upper = get_iqr_bounds(X_train[col])
    outlier_bounds[col] = (lower, upper)

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

    print(f"{col}: capped to [{lower:.2f}, {upper:.2f}]")

meals_served: capped to [-90.88, 714.12]
temperature_C: capped to [-3.82, 48.27]


## 8. Handle Missing Values, Encode, and Scale

Unlike the previous version, the scaler and encoders are fitted and saved as
**two separate files** (`scaler.pkl`, `encoders.pkl`) instead of one combined
`ColumnTransformer` - this is the team's chosen artifact layout.

- **Missing values:** `staff_experience` (164 missing, ~18%, per EDA) is
  imputed with the most frequent category.
- **Encoding:** `staff_experience` and `waste_category` are one-hot encoded.
- **Scaling:** all numeric columns are standardized with `StandardScaler`.

Everything is fit on `X_train` only, then applied to `X_test` with `.transform()`.

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "meals_served",
    "kitchen_staff",
    "temperature_C",
    "humidity_percent",
    "day_of_week",
    "special_event",
    "past_waste_kg"
]

# --- staff_experience: impute missing values, then one-hot encode ---
staff_imputer = SimpleImputer(strategy="most_frequent")
staff_train_imputed = staff_imputer.fit_transform(X_train[["staff_experience"]])
staff_test_imputed = staff_imputer.transform(X_test[["staff_experience"]])

staff_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
staff_train_encoded = staff_encoder.fit_transform(staff_train_imputed)
staff_test_encoded = staff_encoder.transform(staff_test_imputed)
staff_feature_names = staff_encoder.get_feature_names_out(["staff_experience"])

# --- waste_category: no missing values, one-hot encode ---
category_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
category_train_encoded = category_encoder.fit_transform(X_train[["waste_category"]])
category_test_encoded = category_encoder.transform(X_test[["waste_category"]])
category_feature_names = category_encoder.get_feature_names_out(["waste_category"])

# --- numeric columns: scale ---
scaler = StandardScaler()
numeric_train_scaled = scaler.fit_transform(X_train[numeric_features])
numeric_test_scaled = scaler.transform(X_test[numeric_features])

print("Numeric scaled shape (train):", numeric_train_scaled.shape)
print("Staff experience encoded shape (train):", staff_train_encoded.shape)
print("Waste category encoded shape (train):", category_train_encoded.shape)

Numeric scaled shape (train): (728, 7)
Staff experience encoded shape (train): (728, 3)
Waste category encoded shape (train): (728, 4)


## 9. Assemble Processed Feature Tables

Combine the scaled numeric columns and encoded categorical columns back into
a single, human-readable DataFrame with proper column names (not a raw
NumPy array) - this is what gets written to `processed_train.csv` /
`processed_test.csv`.

In [8]:
import pandas as pd

def assemble_processed(numeric_scaled, staff_encoded, category_encoded, index):
    df_out = pd.DataFrame(numeric_scaled, columns=numeric_features, index=index)
    df_out[list(staff_feature_names)] = staff_encoded
    df_out[list(category_feature_names)] = category_encoded
    return df_out

X_train_processed = assemble_processed(
    numeric_train_scaled, staff_train_encoded, category_train_encoded, X_train.index
)
X_test_processed = assemble_processed(
    numeric_test_scaled, staff_test_encoded, category_test_encoded, X_test.index
)

print("Processed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)
X_train_processed.head()

Processed training data shape: (728, 14)
Processed testing data shape: (183, 14)


,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience_beginner,staff_experience_expert,staff_experience_intermediate,waste_category_dairy,waste_category_grains,waste_category_meat,waste_category_vegetables
25,1.132583,-0.220570,-1.140924,-0.109057,1.475478,-0.307794,-0.266106,0.0,0.0,1.0,0.0,0.0,0.0,1.0
84,0.950358,-1.163740,0.257999,0.609742,-0.517375,-0.307794,0.485428,0.0,0.0,1.0,1.0,0.0,0.0,0.0
10,1.307216,0.722601,-0.147202,-0.403326,-0.517375,-0.307794,1.089338,0.0,1.0,0.0,0.0,0.0,1.0,0.0
342,1.375551,0.958393,0.779092,1.497073,-0.019162,-0.307794,1.070441,0.0,1.0,0.0,1.0,0.0,0.0,0.0
889,0.069601,0.251015,0.052447,0.306160,0.977265,-0.307794,1.558598,1.0,0.0,0.0,0.0,0.0,1.0,0.0


## 10. Save Processed Data and Artifacts

- `processed_train.csv` / `processed_test.csv` - model-ready feature tables
  with the target column included, for readability and report screenshots.
- `scaler.pkl` - the fitted `StandardScaler`, plus the numeric column list and
  the outlier bounds learned above (needed so the backend can cap incoming
  values consistently at prediction time).
- `encoders.pkl` - the fitted imputer and one-hot encoders for the
  categorical columns.

These two `.pkl` files are what `feature/ml-api` will load to preprocess a
new prediction request the same way this training data was processed.

In [9]:
import joblib
import os

os.makedirs("../data", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save processed CSVs (with target column)
train_out = X_train_processed.copy()
train_out["food_waste_kg"] = y_train.values
train_out.to_csv("../data/processed_train.csv", index=False)

test_out = X_test_processed.copy()
test_out["food_waste_kg"] = y_test.values
test_out.to_csv("../data/processed_test.csv", index=False)

print("Saved processed_train.csv:", train_out.shape)
print("Saved processed_test.csv:", test_out.shape)

# Save scaler bundle (scaler + outlier bounds + column list)
scaler_bundle = {
    "scaler": scaler,
    "numeric_features": numeric_features,
    "outlier_bounds": outlier_bounds,
}
joblib.dump(scaler_bundle, "../models/scaler.pkl")

# Save encoders bundle (imputer + both encoders)
encoders_bundle = {
    "staff_imputer": staff_imputer,
    "staff_encoder": staff_encoder,
    "category_encoder": category_encoder,
}
joblib.dump(encoders_bundle, "../models/encoders.pkl")

print("Saved scaler.pkl and encoders.pkl to ../models/")

Saved processed_train.csv: (728, 15)
Saved processed_test.csv: (183, 15)
Saved scaler.pkl and encoders.pkl to ../models/


## 11. Sanity Check: Reload Artifacts and Transform a New Row

Confirms `scaler.pkl` / `encoders.pkl` work correctly when loaded fresh from
disk and applied to a single new row - exactly what the FastAPI backend will
do at prediction time.

In [10]:
reloaded_scaler_bundle = joblib.load("../models/scaler.pkl")
reloaded_encoders_bundle = joblib.load("../models/encoders.pkl")

sample_row = X_test.iloc[[0]].copy()

# Apply outlier bounds
for col, (lower, upper) in reloaded_scaler_bundle["outlier_bounds"].items():
    sample_row[col] = sample_row[col].clip(lower, upper)

# Apply staff_experience imputer + encoder
sample_staff_imputed = reloaded_encoders_bundle["staff_imputer"].transform(sample_row[["staff_experience"]])
sample_staff_encoded = reloaded_encoders_bundle["staff_encoder"].transform(sample_staff_imputed)

# Apply waste_category encoder
sample_category_encoded = reloaded_encoders_bundle["category_encoder"].transform(sample_row[["waste_category"]])

# Apply scaler
sample_numeric_scaled = reloaded_scaler_bundle["scaler"].transform(sample_row[reloaded_scaler_bundle["numeric_features"]])

sample_processed = assemble_processed(
    sample_numeric_scaled, sample_staff_encoded, sample_category_encoded, sample_row.index
)

print("Reloaded single-row transform shape:", sample_processed.shape)
sample_processed

Reloaded single-row transform shape: (1, 14)


,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience_beginner,staff_experience_expert,staff_experience_intermediate,waste_category_dairy,waste_category_grains,waste_category_meat,waste_category_vegetables
704,-1.274314,1.429978,-0.748675,0.147754,0.479051,-0.307794,1.155563,0.0,0.0,1.0,0.0,0.0,1.0,0.0
